# Hospital Gold – Capa Oro
Notebook 2 de 2 de la arquitectura medallón del hospital.

| Capa | Ubicación | Tipo de objeto | Propósito |
|---|---|---|---|
| Plata (entrada) | `bi.hospital_silver` | Tablas Delta | Debe existir: se construye en **Hospital Silver** |
| **Oro (este notebook)** | `bi.hospital_gold` | Vistas | Modelo constelación (4 hechos + 7 dimensiones) listo para Power BI |

**Prerrequisito:** haber ejecutado *Hospital Silver* con todas sus validaciones en verde.

## 0. Esquema

In [0]:
CREATE SCHEMA IF NOT EXISTS bi.hospital_gold COMMENT 'Capa oro: modelo dimensional del hospital';

In [0]:
-- Prerrequisito: verifica que la capa plata exista antes de crear las vistas
SHOW TABLES IN bi.hospital_silver;
-- Esperado: 13 tablas

database,tableName,isTemporary
hospital_silver,appointments,false
hospital_silver,bed_records,false
hospital_silver,beds,false
hospital_silver,departments,false
hospital_silver,doctors,false
hospital_silver,helpers,false
hospital_silver,medical_records,false
hospital_silver,nurses,false
hospital_silver,patients,false
hospital_silver,room_records,false


# CAPA ORO
Modelo **constelación** (galaxia): 4 tablas de hechos que comparten dimensiones conformadas.

| Hecho | Grano (1 fila = ...) | Dimensiones |
|---|---|---|
| `fact_appointments` | una cita | paciente, médico, departamento, fecha |
| `fact_admissions` | una estancia (cama o habitación) | paciente, cama, habitación, departamento, personal, fecha ingreso/egreso |
| `fact_surgeries` | una cirugía | paciente, cirujano, habitación, personal, fecha |
| `fact_medical_records` | una consulta clínica | paciente, médico, fecha |

Llaves sustitutas con `ROW_NUMBER()` ordenado por la llave natural (mismo patrón de BI Bike).

## 2.1 Dimensiones

In [0]:
CREATE OR REPLACE VIEW bi.hospital_gold.dim_departments
COMMENT 'Dimensión departamento'
AS
SELECT
    ROW_NUMBER() OVER (ORDER BY dept_id) AS department_key,
    dept_id                              AS department_id,
    dept_name                            AS department_name
FROM bi.hospital_silver.departments;

In [0]:
CREATE OR REPLACE VIEW bi.hospital_gold.dim_patients
COMMENT 'Dimensión paciente con edad y grupo etario calculados al día de hoy'
AS
WITH base AS (
    SELECT *,
           CAST(floor(months_between(current_date(), birth_date) / 12) AS INT) AS age
    FROM bi.hospital_silver.patients
)
SELECT
    ROW_NUMBER() OVER (ORDER BY patient_id) AS patient_key,
    patient_id,
    full_name,
    first_name,
    last_name,
    gender,
    birth_date,
    age,
    CASE
        WHEN age IS NULL THEN 'Sin dato'
        WHEN age < 13    THEN '0-12 Niñez'
        WHEN age < 18    THEN '13-17 Adolescencia'
        WHEN age < 30    THEN '18-29 Joven'
        WHEN age < 45    THEN '30-44 Adulto joven'
        WHEN age < 60    THEN '45-59 Adulto'
        ELSE                  '60+ Adulto mayor'
    END                                     AS age_group,
    city,
    province,
    country
FROM base;

In [0]:
CREATE OR REPLACE VIEW bi.hospital_gold.dim_doctors
COMMENT 'Dimensión médico con su departamento'
AS
SELECT
    ROW_NUMBER() OVER (ORDER BY d.doctor_id) AS doctor_key,
    d.doctor_id,
    concat(d.title, ' ', d.full_name)        AS doctor_name,
    d.gender,
    d.is_surgeon,
    d.surgeon_type,
    d.office_no,
    dep.department_key,
    dep.department_name
FROM bi.hospital_silver.doctors d
LEFT JOIN bi.hospital_gold.dim_departments dep
       ON d.dept_id = dep.department_id;

In [0]:
CREATE OR REPLACE VIEW bi.hospital_gold.dim_staff
COMMENT 'Dimensión personal asistencial: enfermería + auxiliares (role-playing en los hechos)'
AS
WITH staff AS (
    SELECT 'Enfermería' AS role, nurse_id  AS staff_id, full_name, gender, dept_id FROM bi.hospital_silver.nurses
    UNION ALL
    SELECT 'Auxiliar'   AS role, helper_id AS staff_id, full_name, gender, dept_id FROM bi.hospital_silver.helpers
)
SELECT
    ROW_NUMBER() OVER (ORDER BY s.role, s.staff_id) AS staff_key,
    s.role,
    s.staff_id,
    s.full_name                                     AS staff_name,
    s.gender,
    dep.department_key,
    dep.department_name
FROM staff s
LEFT JOIN bi.hospital_gold.dim_departments dep
       ON s.dept_id = dep.department_id;

In [0]:
CREATE OR REPLACE VIEW bi.hospital_gold.dim_beds
COMMENT 'Dimensión cama con la jerarquía física aplanada: departamento → pabellón → cama'
AS
SELECT
    ROW_NUMBER() OVER (ORDER BY b.bed_no) AS bed_key,
    b.bed_no,
    w.ward_no,
    w.ward_name,
    dep.department_key,
    dep.department_name
FROM bi.hospital_silver.beds b
LEFT JOIN bi.hospital_silver.wards w
       ON b.ward_no = w.ward_no
LEFT JOIN bi.hospital_gold.dim_departments dep
       ON w.dept_id = dep.department_id;

In [0]:
CREATE OR REPLACE VIEW bi.hospital_gold.dim_rooms
COMMENT 'Dimensión habitación'
AS
SELECT
    ROW_NUMBER() OVER (ORDER BY r.room_no) AS room_key,
    r.room_no,
    r.room_type,
    dep.department_key,
    dep.department_name
FROM bi.hospital_silver.rooms r
LEFT JOIN bi.hospital_gold.dim_departments dep
       ON r.dept_id = dep.department_id;

In [0]:
CREATE OR REPLACE VIEW bi.hospital_gold.dim_date
COMMENT 'Dimensión calendario generada dinámicamente según el rango de fechas de los hechos'
AS
WITH all_dates AS (
    SELECT appointment_date AS d FROM bi.hospital_silver.appointments
    UNION ALL SELECT admission_date  FROM bi.hospital_silver.bed_records
    UNION ALL SELECT discharge_date  FROM bi.hospital_silver.bed_records
    UNION ALL SELECT admission_date  FROM bi.hospital_silver.room_records
    UNION ALL SELECT discharge_date  FROM bi.hospital_silver.room_records
    UNION ALL SELECT surgery_date    FROM bi.hospital_silver.surgery_records
    UNION ALL SELECT visit_date      FROM bi.hospital_silver.medical_records
    UNION ALL SELECT next_visit_date FROM bi.hospital_silver.medical_records
),
bounds AS (
    SELECT make_date(year(min(d)), 1, 1)   AS start_d,
           make_date(year(max(d)), 12, 31) AS end_d
    FROM all_dates
    WHERE d IS NOT NULL
),
days AS (
    SELECT explode(sequence(start_d, end_d, INTERVAL 1 DAY)) AS full_date
    FROM bounds
)
SELECT
    CAST(date_format(full_date, 'yyyyMMdd') AS INT)  AS date_key,
    full_date,
    year(full_date)                                  AS year,
    quarter(full_date)                               AS quarter,
    concat('T', quarter(full_date))                  AS quarter_name,
    month(full_date)                                 AS month,
    element_at(array('Enero','Febrero','Marzo','Abril','Mayo','Junio','Julio',
                     'Agosto','Septiembre','Octubre','Noviembre','Diciembre'),
               month(full_date))                     AS month_name,
    date_format(full_date, 'yyyy-MM')                AS year_month,
    dayofweek(full_date)                             AS day_of_week,   -- 1 = domingo
    element_at(array('Domingo','Lunes','Martes','Miércoles','Jueves','Viernes','Sábado'),
               dayofweek(full_date))                 AS day_name,
    dayofweek(full_date) IN (1, 7)                   AS is_weekend
FROM days;

## 2.2 Hechos

In [0]:
CREATE OR REPLACE VIEW bi.hospital_gold.fact_appointments
COMMENT 'Hecho cita: 1 fila por cita'
AS
SELECT
    a.appointment_id,
    p.patient_key,
    d.doctor_key,
    d.department_key,
    CAST(date_format(a.appointment_date, 'yyyyMMdd') AS INT) AS date_key,
    a.reason,
    a.status,
    a.appointment_mode,
    a.payment_method,
    a.payment_amount,
    CASE WHEN a.status_code = 'Completed' THEN 1 ELSE 0 END  AS is_completed,
    CASE WHEN a.status_code = 'No-Show'   THEN 1 ELSE 0 END  AS is_no_show,
    CASE WHEN a.status_code = 'Cancelled' THEN 1 ELSE 0 END  AS is_cancelled
FROM bi.hospital_silver.appointments a
LEFT JOIN bi.hospital_gold.dim_patients p ON a.patient_id = p.patient_id
LEFT JOIN bi.hospital_gold.dim_doctors  d ON a.doctor_id  = d.doctor_id;

In [0]:
CREATE OR REPLACE VIEW bi.hospital_gold.fact_admissions
COMMENT 'Hecho estancia: 1 fila por hospitalización, en cama o en habitación'
AS
WITH stays AS (
    SELECT 'Cama' AS stay_type, admission_id, patient_id,
           bed_no, CAST(NULL AS BIGINT) AS room_no,
           nurse_id, helper_id, admission_date, discharge_date,
           length_of_stay_days, is_open, amount, payment_method
    FROM bi.hospital_silver.bed_records
    UNION ALL
    SELECT 'Habitación', admission_id, patient_id,
           CAST(NULL AS BIGINT), room_no,
           nurse_id, helper_id, admission_date, discharge_date,
           length_of_stay_days, is_open, amount, payment_method
    FROM bi.hospital_silver.room_records
)
SELECT
    concat(s.stay_type, '-', s.admission_id)                       AS admission_code,  -- único aunque los ids se repitan entre fuentes
    s.stay_type,
    s.admission_id,
    p.patient_key,
    b.bed_key,
    r.room_key,
    COALESCE(b.department_key, r.department_key)                   AS department_key,
    n.staff_key                                                    AS nurse_key,
    h.staff_key                                                    AS helper_key,
    CAST(date_format(s.admission_date, 'yyyyMMdd') AS INT)         AS admission_date_key,
    CAST(date_format(s.discharge_date, 'yyyyMMdd') AS INT)         AS discharge_date_key,
    s.length_of_stay_days,
    s.is_open,
    s.amount,
    s.payment_method
FROM stays s
LEFT JOIN bi.hospital_gold.dim_patients p ON s.patient_id = p.patient_id
LEFT JOIN bi.hospital_gold.dim_beds     b ON s.bed_no     = b.bed_no
LEFT JOIN bi.hospital_gold.dim_rooms    r ON s.room_no    = r.room_no
LEFT JOIN bi.hospital_gold.dim_staff    n ON n.role = 'Enfermería' AND s.nurse_id  = n.staff_id
LEFT JOIN bi.hospital_gold.dim_staff    h ON h.role = 'Auxiliar'   AND s.helper_id = h.staff_id;

In [0]:
CREATE OR REPLACE VIEW bi.hospital_gold.fact_surgeries
COMMENT 'Hecho cirugía: 1 fila por procedimiento'
AS
SELECT
    s.surgery_id,
    p.patient_key,
    d.doctor_key                                          AS surgeon_key,
    r.room_key,
    COALESCE(r.department_key, d.department_key)          AS department_key,
    n.staff_key                                           AS nurse_key,
    h.staff_key                                           AS helper_key,
    CAST(date_format(s.surgery_date, 'yyyyMMdd') AS INT)  AS date_key,
    s.surgery_type,
    s.start_ts,
    s.end_ts,
    s.duration_minutes,
    s.notes
FROM bi.hospital_silver.surgery_records s
LEFT JOIN bi.hospital_gold.dim_patients p ON s.patient_id = p.patient_id
LEFT JOIN bi.hospital_gold.dim_doctors  d ON s.surgeon_id = d.doctor_id
LEFT JOIN bi.hospital_gold.dim_rooms    r ON s.room_no    = r.room_no
LEFT JOIN bi.hospital_gold.dim_staff    n ON n.role = 'Enfermería' AND s.nurse_id  = n.staff_id
LEFT JOIN bi.hospital_gold.dim_staff    h ON h.role = 'Auxiliar'   AND s.helper_id = h.staff_id;

In [0]:
CREATE OR REPLACE VIEW bi.hospital_gold.fact_medical_records
COMMENT 'Hecho registro clínico: 1 fila por consulta con signos vitales'
AS
SELECT
    m.record_id,
    p.patient_key,
    d.doctor_key,
    d.department_key,
    CAST(date_format(m.visit_date, 'yyyyMMdd') AS INT)       AS date_key,
    CAST(date_format(m.next_visit_date, 'yyyyMMdd') AS INT)  AS next_visit_date_key,
    datediff(m.next_visit_date, m.visit_date)                AS days_to_next_visit,
    m.diagnosis,
    m.treatment,
    m.weight,
    m.height,
    m.systolic_bp,
    m.diastolic_bp,
    m.temp_c,
    CASE WHEN m.systolic_bp >= 140 OR m.diastolic_bp >= 90 THEN 1 ELSE 0 END AS is_high_bp,
    CASE WHEN m.diagnosis RLIKE '(?i)diabet|hypertens|hipertens|asthma|asma' THEN 1 ELSE 0 END AS is_chronic
FROM bi.hospital_silver.medical_records m
LEFT JOIN bi.hospital_gold.dim_patients p ON m.patient_id = p.patient_id
LEFT JOIN bi.hospital_gold.dim_doctors  d ON m.doctor_id  = d.doctor_id;

## 2.3 Validación de oro

In [0]:
-- a) Integridad referencial: hechos con llaves huérfanas (esperado: 0 en todas las columnas)
SELECT 'fact_appointments' AS hecho,
       sum(CASE WHEN patient_key IS NULL THEN 1 ELSE 0 END) AS sin_paciente,
       sum(CASE WHEN doctor_key  IS NULL THEN 1 ELSE 0 END) AS sin_medico,
       sum(CASE WHEN date_key NOT IN (SELECT date_key FROM bi.hospital_gold.dim_date) THEN 1 ELSE 0 END) AS sin_fecha
FROM bi.hospital_gold.fact_appointments
UNION ALL
SELECT 'fact_admissions',
       sum(CASE WHEN patient_key IS NULL THEN 1 ELSE 0 END),
       sum(CASE WHEN bed_key IS NULL AND room_key IS NULL THEN 1 ELSE 0 END),
       sum(CASE WHEN admission_date_key NOT IN (SELECT date_key FROM bi.hospital_gold.dim_date) THEN 1 ELSE 0 END)
FROM bi.hospital_gold.fact_admissions
UNION ALL
SELECT 'fact_surgeries',
       sum(CASE WHEN patient_key IS NULL THEN 1 ELSE 0 END),
       sum(CASE WHEN surgeon_key IS NULL THEN 1 ELSE 0 END),
       sum(CASE WHEN date_key NOT IN (SELECT date_key FROM bi.hospital_gold.dim_date) THEN 1 ELSE 0 END)
FROM bi.hospital_gold.fact_surgeries
UNION ALL
SELECT 'fact_medical_records',
       sum(CASE WHEN patient_key IS NULL THEN 1 ELSE 0 END),
       sum(CASE WHEN doctor_key  IS NULL THEN 1 ELSE 0 END),
       sum(CASE WHEN date_key NOT IN (SELECT date_key FROM bi.hospital_gold.dim_date) THEN 1 ELSE 0 END)
FROM bi.hospital_gold.fact_medical_records;

hecho,sin_paciente,sin_medico,sin_fecha
fact_appointments,0,0,0
fact_admissions,0,0,0
fact_surgeries,0,0,0
fact_medical_records,0,0,0


In [0]:
-- b) Unicidad de llaves sustitutas (esperado: 0 filas)
SELECT 'dim_patients' AS dim, patient_key AS k, count(*) c FROM bi.hospital_gold.dim_patients GROUP BY patient_key HAVING count(*) > 1
UNION ALL SELECT 'dim_doctors', doctor_key, count(*) FROM bi.hospital_gold.dim_doctors GROUP BY doctor_key HAVING count(*) > 1
UNION ALL SELECT 'dim_staff',   staff_key,  count(*) FROM bi.hospital_gold.dim_staff   GROUP BY staff_key  HAVING count(*) > 1
UNION ALL SELECT 'dim_beds',    bed_key,    count(*) FROM bi.hospital_gold.dim_beds    GROUP BY bed_key    HAVING count(*) > 1
UNION ALL SELECT 'dim_rooms',   room_key,   count(*) FROM bi.hospital_gold.dim_rooms   GROUP BY room_key   HAVING count(*) > 1;

dim,k,c


In [0]:
-- c) Reconciliación de negocio: el recaudo de citas en oro debe ser igual al de bronce
SELECT
    (SELECT sum(payment_amount) FROM bi.hospital.appointment)              AS recaudo_bronce,
    (SELECT sum(payment_amount) FROM bi.hospital_gold.fact_appointments)   AS recaudo_oro;
-- En el notebook original: General Medicine 830.436 + Cardiology 269.862 + Neurology 195.473
-- + Pulmonology 184.568 + Endocrinology 183.015 = 1.663.354

recaudo_bronce,recaudo_oro
1663354,1663354.00


## 2.4 Ejemplo de consumo: la misma pregunta de negocio, ahora sobre oro

In [0]:
-- Recaudo y tasa de inasistencia por departamento y trimestre
SELECT
    dep.department_name                         AS departamento,
    dd.year                                     AS anio,
    dd.quarter_name                             AS trimestre,
    count(*)                                    AS total_citas,
    sum(f.payment_amount)                       AS recaudo,
    round(100 * avg(f.is_no_show), 1)           AS pct_no_asistio
FROM bi.hospital_gold.fact_appointments f
JOIN bi.hospital_gold.dim_departments dep ON f.department_key = dep.department_key
JOIN bi.hospital_gold.dim_date        dd  ON f.date_key       = dd.date_key
GROUP BY dep.department_name, dd.year, dd.quarter_name
ORDER BY anio, trimestre, recaudo DESC;

departamento,anio,trimestre,total_citas,recaudo,pct_no_asistio
General Medicine,2024,T3,116,192177.00,12.1
Cardiology,2024,T3,35,60092.00,14.3
Neurology,2024,T3,30,51995.00,3.3
Endocrinology,2024,T3,24,45485.00,0.0
Pulmonology,2024,T3,23,34419.00,8.7
General Medicine,2024,T4,131,216994.00,9.9
Cardiology,2024,T4,44,70979.00,4.5
Endocrinology,2024,T4,34,56062.00,11.8
Pulmonology,2024,T4,24,42554.00,20.8
Neurology,2024,T4,25,42387.00,0.0
